# Kalshi Preliminary EDA
KJ

In [2]:
import requests

# Get series information for KXHIGHNY
url = "https://api.elections.kalshi.com/trade-api/v2/series/KXHIGHNY"
response = requests.get(url)
series_data = response.json()

print(f"Series Title: {series_data['series']['title']}")
print(f"Frequency: {series_data['series']['frequency']}")
print(f"Category: {series_data['series']['category']}")


Series Title: Highest temperature in NYC
Frequency: daily
Category: Climate and Weather


In [ ]:
import requests
import datetime
import base64
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import padding
import os
import sys

# Ensure these are set in your environment or hardcode for testing purposes
API_KEY_ID = os.getenv("KALSHI_ACCESS_KEY")  # Env var uses underscore, NOT dash
PRIVATE_KEY_PATH = 'private-key.key'
BASE_URL = 'https://demo-api.kalshi.co'  # Change to 'https://api.kalshi.com' for production

def load_private_key(key_path):
    """Load the private key from file."""
    with open(key_path, "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())

def create_signature(private_key, timestamp, method, path):
    """Create the request signature."""
    # Remove query params, Kalshi expects only path
    path_without_query = path.split('?')[0]
    message = f'{timestamp}{method}{path_without_query}'.encode("utf-8")
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH
        ),
        hashes.SHA256()
    )
    return base64.b64encode(signature).decode("utf-8")

def get_bal(private_key, api_key_id, path, base_url=BASE_URL):
    """Make an authenticated GET request to Kalshi API."""
    timestamp = str(int(datetime.datetime.now(datetime.timezone.utc).timestamp() * 1000))
    signature = create_signature(private_key, timestamp, "GET", path)

    headers = {
        "Kalshi-Access-Key": api_key_id,
        "Kalshi-Access-Timestamp": timestamp,
        "Kalshi-Access-Signature": signature
    }
    response = requests.get(base_url + path, headers=headers)
    return response

# --- MAIN LOGIC ---

# Sanity checks
if API_KEY_ID is None:
    print("ERROR: Kalshi API key was not found in your environment variable 'KALSHI_ACCESS_KEY'.")
    print("Set it with e.g. `export KALSHI_ACCESS_KEY=your_key_here` and restart the notebook.")
    sys.exit(1)

try:
    private_key = load_private_key(PRIVATE_KEY_PATH)
except Exception as e:
    print(f"ERROR: Could not load private key from {PRIVATE_KEY_PATH}: {e}")
    sys.exit(1)

balance_path = "/trade-api/v2/portfolio/balance"
response = get_bal(private_key, API_KEY_ID, balance_path)
try:
    data = response.json()
except Exception as e:
    print("ERROR: Could not parse JSON response from Kalshi:", e)
    print("Raw response text:")
    print(response.text)
    sys.exit(1)

if "error" in data:
    print("Authentication or API error encountered:")
    print(data["error"].get("message", "Unknown error"))
    print(data)
else:
    print(data)


Authentication or API error encountered:
authentication_error
{'error': {'code': 'authentication_error', 'message': 'authentication_error', 'details': 'NOT_FOUND'}}
